# Remote Server Tutorial

> **Note:** Code cells in this notebook contain shell commands to run in a terminal — they are not meant to be executed as notebook cells. Commands labeled **\[local\]** should be run on your own machine; commands labeled **\[server\]** should be run after SSH-ing into the server.

---

## 0. Prerequisites

Before you start, make sure you have the following on your local machine:

- **CMU VPN** — you must be connected to the VPN to reach the server. Install it here if needed:  
  https://www.cmu.edu/computing/services/endpoint/network-access/vpn/how-to/index.html
- **VS Code** with the **Remote - SSH** extension:
  1. Open the Extensions sidebar (`Cmd+Shift+X` / `Ctrl+Shift+X`)
  2. Search for **Remote - SSH** (publisher: Microsoft, extension ID `ms-vscode-remote.remote-ssh`)
  3. Click Install
- **Git** installed
- **Python 3** installed
- Your Andrew ID and password

---
## 1. Initial Connection

Connect to the server using your Andrew ID and password. This first login uses password authentication — we'll switch to SSH key authentication in the next section.

**\[local\]**

In [ ]:
ssh YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu

After entering your password, the prompt will change to something like:

```
YOUR_ANDREW_ID@heinz-90803:~$
```

You are now on the server. Commands you type here run on the remote machine, not on your laptop.

When you're ready to move on to SSH key setup, type `exit` to disconnect.

---
## 2. SSH Key Setup

SSH keys let you log in without typing a password each time. You have a key **pair**: a public key (`.pub`) that you can share freely, and a private key (no extension) that you must never share with anyone.

### 2.1 Check for an Existing Key

**\[local\]** — run this to see if you already have an SSH key pair:

In [ ]:
ls ~/.ssh/*.pub

If you see `id_ed25519.pub` or `id_rsa.pub`, you already have a key pair — skip to Section 2.2.

If no key exists, generate one **\[local\]**:

In [ ]:
ssh-keygen -t ed25519 -C "YOUR_ANDREW_ID@andrew.cmu.edu"

Press Enter to accept the default save path (`~/.ssh/id_ed25519`). You can optionally add a passphrase. This creates two files:

- `~/.ssh/id_ed25519` — your **private** key. Never share this file.
- `~/.ssh/id_ed25519.pub` — your **public** key. Safe to share.

### 2.2 Copy Your Public Key to the Server with `scp`

`scp` (secure copy) transfers files over SSH using the same credentials and host addressing. Its syntax mirrors `cp`, but uses `user@host:path` for remote locations:

```
scp <source> <destination>
```

Copy your public key from your local machine to the server **\[local\]**:

In [ ]:
scp ~/.ssh/id_ed25519.pub YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu:~/.ssh/uploaded_key.pub

Now log in to the server with your password and add the key to `authorized_keys` **\[server\]**:

In [ ]:
mkdir -p ~/.ssh
chmod 700 ~/.ssh
cat ~/.ssh/uploaded_key.pub >> ~/.ssh/authorized_keys
chmod 600 ~/.ssh/authorized_keys

Disconnect and reconnect — you should no longer be prompted for a password **\[local\]**:

In [ ]:
exit
ssh YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu

### 2.3 SSH Config: Give the Server a Nickname

Typing `heinz-90803.heinz.cmu.edu` every time is tedious, and when you're managing multiple servers it becomes unmanageable fast. An SSH config file maps short aliases to full connection details.

Open your config file in a text editor **\[local\]**:

In [ ]:
nano ~/.ssh/config

Add this block, replacing `YOUR_ANDREW_ID`:

```
Host heinz
    HostName heinz-90803.heinz.cmu.edu
    User YOUR_ANDREW_ID
    IdentityFile ~/.ssh/id_ed25519
    ServerAliveInterval 60
    ServerAliveCountMax 5
```

What each field does:
- `Host heinz` — the alias you type at the command line
- `HostName` — the real server address
- `User` — your login name, so you don't need to type `andrew_id@` each time
- `IdentityFile` — which private key to use for authentication
- `ServerAliveInterval` / `ServerAliveCountMax` — sends keepalive pings so idle connections aren't dropped

In `nano`, save with `Ctrl+O` then Enter, and exit with `Ctrl+X`.

Now connect with just **\[local\]**:

In [ ]:
ssh heinz

The alias works with `scp` too:

```bash
scp somefile.csv heinz:~/some/path/
```

### 2.4 Connect with VS Code Remote-SSH

With the Remote - SSH extension installed:

1. Open the Command Palette (`Cmd+Shift+P` / `Ctrl+Shift+P`)
2. Type **Remote-SSH: Connect to Host** and select it
3. Select `heinz` from the list (or type `YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu`)
4. A new VS Code window opens — the file explorer and integrated terminal are now on the server
5. Use **File → Open Folder** to open a directory on the server

Inside this window, saving a file saves it directly to the server.

---
## 3. Add Your Server's SSH Key to GitHub

To push code from the server to GitHub, GitHub needs to authenticate the server. The cleanest way is SSH: you generate a key pair on the server, then register the public key in your GitHub account. This is separate from the key you use to log in to the server — each machine that communicates with GitHub needs its own entry.

**\[server\]** — check whether the server already has an SSH key:

In [ ]:
ls ~/.ssh/*.pub

If you see a `.pub` file, the server already has a key — skip ahead to printing it. Otherwise, generate one:

In [ ]:
ssh-keygen -t ed25519 -C "YOUR_ANDREW_ID@andrew.cmu.edu"

Accept the default path and skip the passphrase (press Enter twice). Now print the public key:

In [ ]:
cat ~/.ssh/id_ed25519.pub

Copy the entire output (it starts with `ssh-ed25519`). Then add it to your GitHub account:

1. Go to **GitHub → Settings → SSH and GPG keys**
2. Click **New SSH key**
3. Give it a descriptive title like `heinz server`
4. Paste the public key into the Key field
5. Click **Add SSH key**

Test that GitHub recognizes the server **\[server\]**:

In [ ]:
ssh -T git@github.com

You should see: `Hi YOUR_GITHUB_USERNAME! You've successfully authenticated...`

If prompted about the host's authenticity, type `yes` to continue.

---
## 4. Fork and Clone the Repo

First, **fork** the repo on GitHub so you have your own copy to push to:

1. Go to the repo on GitHub
2. Click **Fork** (top-right)
3. Accept the defaults and click **Create fork**

Then clone **your fork** on the server **\[server\]**:

In [ ]:
git clone git@github.com:YOUR_GITHUB_USERNAME/remote_server_tutorial_demo.git
cd remote_server_tutorial_demo

---
## 5. Set Up a Virtual Environment

The first thing to do after cloning is create a Python virtual environment. On a shared server, everyone uses the same system Python, so installing packages globally creates version conflicts between users. A virtual environment is isolated to your project directory.

> **Note:** If you ever need to install a Python package outside a venv on this server, add `--user` to the `pip` command (e.g., `pip install somepackage --user`). Inside a venv this flag is not needed.

**\[server\]**

In [ ]:
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

When the environment is active, your prompt shows `(.venv)` at the start. To deactivate:

```bash
deactivate
```

Remember to run `source .venv/bin/activate` again any time you open a new terminal session on the server.

---
## 6. Running Python Code

There are two ways to run Python code on the server.

### 6.1 Command Line

Make sure your venv is active, then run **\[server\]**:

In [ ]:
python src/hello_remote.py

### 6.2 Jupyter Notebook in VS Code

With VS Code Remote-SSH you can open `.ipynb` notebook files directly — the notebook runs on the server with no SSH tunnel needed.

Before opening a notebook, register your virtual environment as a Jupyter kernel so VS Code can use it **\[server\]**:

In [ ]:
python -m ipykernel install --user --name .venv --display-name "Python (.venv)"

Open `remote_demo.ipynb` in VS Code. When prompted to select a kernel, choose **Python (.venv)**.

Run the cells one by one. The last cell imports `tqdm` — you should get a `ModuleNotFoundError` since it isn't installed in your venv yet. Try installing it yourself and re-run the cell.

---
## 7. Create Your Pet Data File

On your **local machine**, create a CSV file with three pieces of information: your name, an unusual pet animal (no cats or dogs), and a color for that animal.

**\[local\]** — fill in your own values before running:

In [ ]:
printf 'name,animal,color\nYOUR_NAME,YOUR_ANIMAL,YOUR_COLOR\n' > pet_data.csv
cat pet_data.csv

For example, if your name is Alice with a pink axolotl:

```bash
printf 'name,animal,color\nAlice,axolotl,pink\n' > pet_data.csv
```

Transfer the file to the server with `scp` **\[local\]**:

In [ ]:
scp pet_data.csv heinz:~/remote_server_tutorial_demo/data/

Verify it arrived on the server **\[server\]**:

In [ ]:
ls -lah data/
cat data/pet_data.csv

---
## 8. Name Your Pet

The server has a shared word list at `/home/shared_data/`. The script `src/name_your_pet.py` reads your `pet_data.csv` and uses those files to generate a unique name for your pet. The result is deterministic — the same inputs always produce the same name, and different students with different inputs get different names.

**\[server\]**

In [ ]:
python src/name_your_pet.py

Ask your neighbor what name they got!

---
## 9. Jupyter Notebook via SSH Tunnel

To run Jupyter on the server but use it in your local browser, you need an SSH tunnel. The tunnel forwards a port from the server to your local machine, so your browser connects to `localhost` but the traffic is routed over SSH to the server.

**Step 1** — start Jupyter on the server **\[server\]**:

In [ ]:
jupyter notebook --no-browser --port 8888

**Step 2** — in a **new local terminal**, open the tunnel **\[local\]**:

In [ ]:
ssh -L 8888:localhost:8888 heinz

The `-L 8888:localhost:8888` flag means: "forward my local port 8888 through the SSH connection to port 8888 on the server."

**Step 3** — open `http://localhost:8888` in your browser. You'll see the Jupyter file browser showing files on the server.

> If you're using VS Code Remote-SSH with the Jupyter extension, VS Code handles port forwarding automatically — you may not need the manual tunnel step.

---
## 10. tmux: Keeping Jobs Running After Disconnect

### Part A: Without tmux

First, see what happens when you run a long job and close your connection.

Start the job **\[server\]**:

In [ ]:
python long_job.py

While it's running, **fully close your terminal window** (or type `exit` to end the SSH session). Then reconnect **\[local\]**:

In [ ]:
ssh heinz

The job is gone. When your SSH session ended, the shell and all its child processes — including `long_job.py` — were killed automatically.

### Part B: With tmux

`tmux` is a terminal multiplexer. It keeps a session running on the server independently of your SSH connection. You can detach from it, disconnect, come back later, and find everything exactly where you left it.

Start a named tmux session and launch the job inside it **\[server\]**:

In [ ]:
tmux new -s demo
python long_job.py

While the job is running, **detach** from the session (this leaves it running in the background on the server):

```
Ctrl-b  then  d
```

You're back at the normal server prompt. The job is still running. Now fully disconnect **\[server\]**:

In [ ]:
exit

Reconnect and find your session **\[local → server\]**:

In [ ]:
ssh heinz
tmux ls              # list all running sessions
tmux attach -t demo  # reattach to the session named "demo"

The job kept running while you were away. When it finishes, kill the session **\[server\]**:

In [ ]:
tmux kill-session -t demo

---
## 11. Commit and Push

Add your `pet_data.csv` to the repo and push it to your fork.

**\[server\]**

In [ ]:
git add data/pet_data.csv
git status
git commit -m "Add pet data for YOUR_ANDREW_ID"
git push origin main

Go to your fork on GitHub and confirm your commit appears on `main`.

---
## 12. Helpful Reference Commands

### Navigation **\[server\]**

In [ ]:
pwd           # print current directory
ls -lah       # list files with sizes and permissions
cd ~          # go to home directory
cd ..         # go up one level
cd -          # go to previous directory
mkdir mydir   # create a directory
cp a b        # copy file a to b
mv a b        # move or rename file a to b
rm file       # delete a file
cat file      # print file contents
head file     # first 10 lines
tail -f file  # follow a file as it grows (useful for logs)

### Monitor Resource Usage

In [ ]:
htop   # interactive process viewer — press q to quit
top    # fallback if htop is unavailable

### Redirect Output to a Log File

In [ ]:
python long_job.py > job.log 2>&1  # redirect stdout and stderr to a file
tail -f job.log                     # watch the output as it arrives

### Text Editors

In [ ]:
nano file.txt  # beginner-friendly: Ctrl+O to save, Ctrl+X to exit
vim file.txt   # powerful, steeper learning curve: i to insert, Esc then :wq to save and quit

### Good Practices on a Shared Server

- Do not leave unnecessary jobs running
- Check CPU and memory usage before launching heavy jobs (`htop`)
- Do not edit, move, or delete other users' files
- Keep your work organized inside your own home directory